# ML Project / NASDAQ Prediction and Analysis (big dataset)

Same pipeline as `1_Project_Features.ipynb`, but starting in 2005 instead of 2022. The NQ 1-minute file (Kaggle) only covers Dec 2022 - Dec 2025, so it is replaced by the Asian indices (Nikkei 225, Hang Seng), which close before our 09:00 ET cutoff and go back decades on yfinance. Bitcoin is dropped (it only starts in 2014).

**Objective** : Predicting if the Nasdaq-100 (`^NDX`) goes up or down **during the New York session of day `t`** (from the open at 09:30 ET to the close at 16:00 ET), using only information known **before 09:00 ET of day `t`**. The overnight information is how Asia (Tokyo, Hong Kong) traded on day `t`; the daily data (stocks, macro) is used as context from the day before.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import requests 
import io
import os
from sklearn.model_selection import TimeSeriesSplit

In [ ]:
path_to_data = ("./data")

## Data

We take the current Nasdaq-100 list from Wikipedia. This means we use today's companies for the whole period, so there's some survivorship bias (we'll mention it in the report).

With 20 years of history this bias is bigger: only the companies that survived until today are in the list.

In [ ]:
html = requests.get("https://en.wikipedia.org/wiki/List_of_NASDAQ-100_companies", headers={"User-Agent": "Mozilla/5.0"}).text
table = pd.read_html(io.StringIO(html))
ndx = next(t for t in table if "Ticker" in t.columns)
TICKERS = ndx["Ticker"].str.replace(".", "-", regex=False).tolist()

In [ ]:
TICKERS

On top of the stocks we add some macro data: volatility indices (VIX, VXN), treasury yields, oil, gold, silver, the dollar and Bitcoin. The idea is to give the model some context about the market, not just prices.

In [ ]:
NASDAQ_FEATURES = {
    # --- INDICES ET VOLATILITÉ ---                    
    "^IXIC": "Nasdaq_composite",              # Broad tech market index
    "^VXN": "Nasdaq_volatility",              # Nasdaq-100 implied volatility (Tech fear index)
    "^VIX": "Volatility",                     # S&P 500 implied volatility (Market fear index)

    # --- interest ---
    "^IRX": "3month_treasury_yield",          # 3-month Treasury yield
    "^FVX": "5y_treasury_yield",              # 5-year Treasury yield
    "^TNX": "10y_treasury_yield",             # 10-year Treasury yield (Tech discounting rate)
    "^TYX": "30y_treasury_yield",             # 30-year Treasury yield

    # --- MATIÈRES PREMIÈRES ET DEVISES ---
    "DX-Y.NYB": "US_dollar",                  # US dollar index (DXY)
    "CL=F": "crude_oil",                      # WTI crude oil futures
    "GC=F": "gold",                           # Gold futures
    "SI=F": "silver",                         # Silver futures
    # ponytail: BTC-USD removed, it starts in 2014 and the final dropna would cut the data back to 2014
}
tickers1 = list(NASDAQ_FEATURES.keys())

Asian indices. Tokyo closes at 15:30 JST (01:30 / 02:30 ET) and Hong Kong at 16:00 HKT (03:00 / 04:00 ET), so the close of day `t` is known before the 09:00 ET cutoff. Europe closes at 11:30 ET, after the cutoff, so we don't use it.

In [ ]:
ASIA = {
    "^N225": "Nikkei_225",
    "^HSI": "Hang_Seng",
}
tickers2 = list(ASIA.keys())

In [ ]:
TICKERS = list(dict.fromkeys(TICKERS + tickers1 + tickers2))

In [ ]:
df_nasdaq = yf.download("^NDX", start="2018-01-01", end="2025-12-12",
                     auto_adjust=True, progress=False)[['Open', 'Close','Volume']]
print(df_nasdaq.columns)
print(df_nasdaq.tail())

In [ ]:
df_nasdaq.head(20)

In [ ]:
raw = yf.download(TICKERS, start="2018-01-01", end="2025-12-12")[['Open', 'Close','Volume']]

In [ ]:
raw.head(10)

We join everything on the Nasdaq-100 dates (inner join). Bitcoin trades on weekends but the index doesn't, so this way we only keep real trading days and we don't need to fill anything.

In [ ]:
raw = pd.concat((df_nasdaq,raw),axis=1,join='inner',)

In [ ]:
raw

### Missing values

Many companies are recent compared to 2005 (ARM, ABNB, PLTR...), so they have no history for most of the period. We drop the columns with less than 40% of data. Later the stock returns only keep the stocks with the full history.

We also drop GOOG (almost the same as GOOGL) and ^IXIC (it moves with the Nasdaq-100), they don't add anything new.

In [ ]:
dropping = raw.isnull().sum().sort_values(ascending=False)/len(raw)

In [ ]:
dropping

In [ ]:
raw_drop = raw.dropna(axis=1, thresh=int(0.4 * len(raw)))
# GOOG is basically GOOGL and ^IXIC moves with ^NDX, so they add nothing
raw_drop = raw_drop.drop(columns=["GOOG", "^IXIC"], level=1)

In [ ]:
print(list(set(raw.columns.tolist()) - set(raw_drop.columns.tolist())))

In [ ]:
try:
    raw_drop.loc[:, (slice(None), "SPCX")]
except KeyError:
    print("SPCX not in raw_drop: it was dropped")

In [ ]:
raw.loc[:, (slice(None), "SPCX")]

In [ ]:
raw_drop.describe()

From the date we keep only `Dayofweek`. The other date columns (year, month, elapsed...) are trends or noise for this target.

In [ ]:
raw_drop = raw_drop.reset_index()
raw_time = raw_drop.copy()
# ponytail: only Dayofweek was kept from fastai add_datepart, so no fastai needed
raw_time[("Dayofweek", "")] = raw_time[("Date", "")].dt.dayofweek

In [ ]:
raw_time.isnull().sum().sort_values(ascending=False)/len(raw)

In [ ]:
columns = raw_time.columns.to_list()
print(columns)

In [ ]:
raw_time = raw_time.copy(True)

In [ ]:
raw_time.loc[:, (slice(None),'^NDX')]

## Target

We decide at 09:00 ET of day `t`, we enter at the NY open of day `t` (09:30) and we exit at the NY close of the same day (16:00).

The target is the open-to-close log return of `^NDX`, and the label is 1 if it is positive, 0 if not. It is open-to-close on purpose: the overnight move is already in the open price. The `Open` and `Close` of `^NDX` of day `t` are only used here, never as features.

In [ ]:
ndx = raw_drop.set_index('Date')
close = ndx['Close']['^NDX']
open_ = ndx['Open']['^NDX']

In [ ]:
open_

In [ ]:
ret_ny = np.log(close / open_)

In [ ]:
ret_ny

In [ ]:
raw_time = raw_time.set_index("Date")

In [ ]:
raw_time

In [ ]:
data = raw_time.copy()
# both have the Date index, so pandas aligns them by date
data["ret_ny"] = ret_ny

In [ ]:
r = data["ret_ny"]
data["y_ny"] = (r > 0).astype(float).where(r.notna())

In [ ]:
print(data[["ret_ny", "y_ny"]].tail(7))

In [ ]:
print(data.columns.tolist())
print(data.index.min(), data.index.max(), len(data))

The target doesn't look ahead anymore, so the last rows keep their target. We only remove the rows without target (if any).

In [ ]:
target_cols = [("ret_ny", ""), ("y_ny", "")]

data = data.dropna(subset=[("y_ny", "")])

## Daily features

Context from the day before. All the daily US data closes at 16:00 or later, so everything here is shifted by 1 day (`shift(1)`): for day `t` we only use values up to the close of `t-1`.

Nasdaq-100: momentum over 1, 5 and 20 days, the NY session return of the day before, and the 20-day volatility of the daily returns.

In [ ]:
ndx_close = data[("Close", "^NDX")]
for n in (1, 5, 20):
    data[(f"ndx_ret_{n}d", "")] = np.log(ndx_close / ndx_close.shift(n)).shift(1)

In [ ]:
data[("ndx_ret_ny_prev", "")] = data[("ret_ny", "")].shift(1)
data[("ndx_vol_20d", "")] = np.log(ndx_close).diff().rolling(20).std().shift(1)

Macro: level for the volatility indices and the yields, daily log return for the dollar, oil and gold. Some of them don't trade on a few Nasdaq days (bond holidays), so we forward fill first (it only uses the past). Silver and the 5y / 30y yields are not used.

In [ ]:
macro_close = data["Close"][["^VIX", "^VXN", "^TNX", "^IRX", "DX-Y.NYB", "CL=F", "GC=F"]].ffill()

In [ ]:
data[("vix", "")] = macro_close["^VIX"].shift(1)
data[("vxn", "")] = macro_close["^VXN"].shift(1)
data[("yield_10y", "")] = macro_close["^TNX"].shift(1)
data[("yield_3m", "")] = macro_close["^IRX"].shift(1)

In [ ]:
data[("dollar_ret", "")] = np.log(macro_close["DX-Y.NYB"]).diff().shift(1)
data[("oil_ret", "")] = np.log(macro_close["CL=F"]).diff().shift(1)
data[("gold_ret", "")] = np.log(macro_close["GC=F"]).diff().shift(1)

### Asia (day `t`)

Log return of the Nikkei and the Hang Seng from their previous close to their close of day `t`. **No shift**: these markets close before 09:00 ET of day `t`. If Asia is closed on day `t` (holiday) we forward fill, so the return is 0.

In [ ]:
asia_close = data["Close"][tickers2].ffill()
data[("nikkei_ret", "")] = np.log(asia_close["^N225"]).diff()
data[("hsi_ret", "")] = np.log(asia_close["^HSI"]).diff()
asia_cols = [("nikkei_ret", ""), ("hsi_ret", "")]

In [ ]:
# anti-leakage: Tokyo and Hong Kong close of day t is before 09:00 ET of day t (in any season)
assert (pd.Timestamp("2025-01-15 15:30", tz="Asia/Tokyo").tz_convert("America/New_York").hour < 9)
assert (pd.Timestamp("2025-07-15 16:00", tz="Asia/Hong_Kong").tz_convert("America/New_York").hour < 9)

Stocks: daily log return of each stock, shifted by 1. We keep only the stocks with no missing values in the period. They are kept for a PCA that we will do later, fitted on train only.

In [ ]:
not_stocks = tickers1 + tickers2 + ["^NDX"]
stocks = [t for t in data["Close"].columns if t not in not_stocks]
stock_close = data["Close"][stocks].dropna(axis=1)
print(len(stocks), "stocks,", stock_close.shape[1], "with no missing values")

In [ ]:
stock_ret = np.log(stock_close).diff().shift(1)
stock_ret.columns = pd.MultiIndex.from_product([["stock_ret"], stock_ret.columns])

Breadth: % of those stocks with a positive return the day before.

In [ ]:
data[("pct_up", "")] = ((stock_ret > 0).mean(axis=1) * 100).where(stock_ret.notna().all(axis=1))

## Join

The Open, Close and Volume levels are not features (they are replaced by the returns above), so we drop them. Then we do an inner join of the daily table and the NQ features on the `^NDX` trading dates.

In [ ]:
data = data.drop(columns=["Open", "Close", "Volume"], level=0)
data = pd.concat([data, stock_ret], axis=1)

The first rows have no 20-day values (warm-up), so they have missing values. We remove the rows with missing values.

In [ ]:
nulls = data.isnull().sum()
print(nulls[nulls > 0].sort_values(ascending=False))

In [ ]:
n_before = len(data)
data = data.dropna()
print("rows dropped:", n_before - len(data), "| first day:", data.index.min().date())

In [ ]:
# anti-leakage: no price level of day t in the features
feature_cols = [c for c in data.columns if c not in target_cols]
assert ("Close", "^NDX") not in feature_cols and ("Open", "^NDX") not in feature_cols
assert not data.columns.get_level_values(0).isin(["Open", "Close", "Volume"]).any()

In [ ]:
print(data.shape)
print(data.index.min(), data.index.max())

In [ ]:
data

In [ ]:
data.to_feather("../data/final_data_ny_big.feather")

## Train / test split

Same as `Data_Splitting.ipynb`: chronological, the test set is the last 20% of the rows. Only used here for the checks.

In [ ]:
split = int(0.8 * len(data))
train, test = data.iloc[:split], data.iloc[split:]

Class balance: see below. A model that always says "up" already gets that accuracy, that's the baseline to beat.

**Known limitation:** the overnight features are only daily closes of Asia, much less detailed than the NQ 1-minute bars of `1_Project_Features.ipynb`. The period also mixes very different markets (2008, 2020, 2022).

## Checks

In [ ]:
print("final shape:", data.shape)
print("train:", train.shape, train.index.min().date(), "->", train.index.max().date())
print("test: ", test.shape, test.index.min().date(), "->", test.index.max().date())

In [ ]:
print([c for c in feature_cols if c[0] != "stock_ret"])
print((data.columns.get_level_values(0) == "stock_ret").sum(), "stock_ret columns")

In [ ]:
print(train[("y_ny", "")].value_counts(normalize=True))
print(test[("y_ny", "")].value_counts(normalize=True))

Leakage sanity check: correlation of each feature with the target return on train (Asia features first, then the daily features). A value above 0.3 (in absolute value) would be suspicious.

In [ ]:
corr = train[asia_cols].corrwith(train[("ret_ny", "")])
corr.index = [c[0] for c in corr.index]
print(corr.sort_values())
print("max |corr|:", corr.abs().max())

In [ ]:
daily_cols = [c for c in feature_cols if c[0] != "stock_ret" and c not in asia_cols]
corr_daily = train[daily_cols].corrwith(train[("ret_ny", "")])
corr_daily.index = [c[0] for c in corr_daily.index]
print(corr_daily.sort_values())
print("max |corr|:", corr_daily.abs().max())